# S2 Cloud Score Threshold Visualizer

Interactive tool to determine an appropriate cloud masking threshold for S2 composites.

- **Bands 0-5**: B2, B3, B4, B8A, B11, B12
- **Band 6**: `mean_cs` (mean cloud score, 0-10000, higher = clearer)
- **Band 7**: `median_cs` (median cloud score, 0-10000, higher = clearer)

Images are **downsampled 3x** (990→330) via block averaging to match the dataloader pipeline. Cloud scores are block-averaged the same way, then thresholded. Pixels with cloud score **below** the threshold are masked (shown in red).

In [16]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ipywidgets import interact, IntSlider, Dropdown, fixed
from IPython.display import display

S2_PATH = '/projectnb/hlsfm/applications/lsp/outputs/S2_composites_HP-LSP'
N_SAMPLES = 10
SEED = 66

In [17]:
# Sample N_SAMPLES random tiles, downsample 3x to match dataloader
all_files = sorted([f for f in os.listdir(S2_PATH) if f.endswith('.tif')])
rng = np.random.RandomState(SEED)
sample_files = list(rng.choice(all_files, size=N_SAMPLES, replace=False))

def downsample_3x(img):
    """Block-average all bands from 990x990 to 330x330."""
    C, H, W = img.shape
    new_h, new_w = H // 3, W // 3
    return img[:, :new_h*3, :new_w*3].reshape(C, new_h, 3, new_w, 3).mean(axis=(2, 4))

# Pre-load and downsample all sampled tiles
tiles = []
for f in sample_files:
    with rasterio.open(os.path.join(S2_PATH, f)) as src:
        img = src.read()  # (8, 990, 990)
    tiles.append(downsample_3x(img))  # (8, 330, 330)
    
print(f'Loaded {len(tiles)} tiles (downsampled to {tiles[0].shape[1]}x{tiles[0].shape[2]})')
for f in sample_files:
    print(f'  {f}')

Loaded 10 tiles (downsampled to 330x330)
  S2_composite_2019-01-01_WY-2_T12TWP.tif
  S2_composite_2019-10-01_NM-7_T13SCS.tif
  S2_composite_2019-10-01_VA-4_T18STH.tif
  S2_composite_2019-05-01_AK-1_T06VWR.tif
  S2_composite_2019-07-01_KS-1_T15SUD.tif
  S2_composite_2019-01-01_ND-2_T14TMT.tif
  S2_composite_2019-09-01_AZ-6_T12SWA.tif
  S2_composite_2020-08-01_TN-1_T17SKV.tif
  S2_composite_2019-12-01_AR-1_T15SYV.tif
  S2_composite_2020-06-01_WY-1_T12TWQ.tif


In [18]:
def make_rgb(img, percentile_clip=2):
    """Create RGB from S2 bands: R=B4(idx2), G=B3(idx1), B=B2(idx0)."""
    rgb = np.stack([img[2], img[1], img[0]], axis=-1).astype(np.float32)
    # Clip outliers and normalize to [0, 1]
    lo = np.percentile(rgb[rgb > 0], percentile_clip) if (rgb > 0).any() else 0
    hi = np.percentile(rgb[rgb > 0], 100 - percentile_clip) if (rgb > 0).any() else 1
    rgb = np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)
    return rgb


def visualize(threshold, score_band):
    """Display all sampled tiles with cloud mask at given threshold."""
    band_idx = 6 if score_band == 'mean_cs' else 7
    
    fig, axes = plt.subplots(2, 5, figsize=(25, 11))
    axes = axes.flatten()
    
    for i, (img, fname) in enumerate(zip(tiles, sample_files)):
        ax = axes[i]
        rgb = make_rgb(img)
        
        # Cloud mask: pixels with score BELOW threshold are cloudy
        cloud_score = img[band_idx].astype(np.float32)
        cloudy = cloud_score < threshold
        # Also ignore nodata (all bands zero)
        nodata = (img[:6] == 0).all(axis=0)
        cloudy = cloudy & ~nodata
        
        # Overlay: tint masked pixels red
        overlay = rgb.copy()
        overlay[cloudy, 0] = 1.0  # red channel
        overlay[cloudy, 1] *= 0.3
        overlay[cloudy, 2] *= 0.3
        # Nodata pixels as dark gray
        overlay[nodata] = 0.15
        
        pct_masked = 100 * cloudy.sum() / (~nodata).sum() if (~nodata).sum() > 0 else 0
        
        ax.imshow(overlay)
        ax.set_title(f'{fname.split("S2_composite_")[1].split(".tif")[0]}\n{pct_masked:.1f}% masked', fontsize=9)
        ax.axis('off')
    
    legend_patches = [
        mpatches.Patch(color='red', label=f'Cloudy ({score_band} < {threshold})'),
        mpatches.Patch(color='gray', label='Nodata'),
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=2, fontsize=12)
    fig.suptitle(f'{score_band} threshold = {threshold}', fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.show()

In [ ]:
interact(
    visualize,
    threshold=IntSlider(value=3000, min=0, max=10000, step=100,
                        description='Threshold:', style={'description_width': 'initial'},
                        layout={'width': '600px'}, continuous_update=False),
    score_band=Dropdown(options=['mean_cs', 'median_cs'], value='mean_cs',
                        description='Score band:'),
);

interactive(children=(IntSlider(value=3000, continuous_update=False, description='Threshold:', layout=Layout(w…

## Per-pixel cloud score distributions

In [ ]:
# Plot histograms of cloud scores across all sampled tiles
mean_cs_all = np.concatenate([img[6].flatten() for img in tiles])
median_cs_all = np.concatenate([img[7].flatten() for img in tiles])

# Exclude zeros (nodata)
mean_cs_valid = mean_cs_all[mean_cs_all != 0]
median_cs_valid = median_cs_all[median_cs_all != 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.hist(mean_cs_valid, bins=100, color='steelblue', alpha=0.7)
ax1.set_title('mean_cs distribution')
ax1.set_xlabel('Cloud score (0=cloudy, 10000=clear)')
ax1.axvline(3000, color='red', ls='--', label='3000')
ax1.axvline(5000, color='orange', ls='--', label='5000')
ax1.legend()

ax2.hist(median_cs_valid, bins=100, color='darkorange', alpha=0.7)
ax2.set_title('median_cs distribution')
ax2.set_xlabel('Cloud score (0=cloudy, 10000=clear)')
ax2.axvline(3000, color='red', ls='--', label='3000')
ax2.axvline(5000, color='orange', ls='--', label='5000')
ax2.legend()

plt.tight_layout()
plt.show()

print('Percentiles:')
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f'  p{p:02d}: mean_cs={np.percentile(mean_cs_valid, p):.0f}  median_cs={np.percentile(median_cs_valid, p):.0f}')

In [ ]:
# Resample different tiles
def resample(seed=None):
    global tiles, sample_files
    rng = np.random.RandomState(seed)
    sample_files = list(rng.choice(all_files, size=N_SAMPLES, replace=False))
    tiles = []
    for f in sample_files:
        with rasterio.open(os.path.join(S2_PATH, f)) as src:
            tiles.append(downsample_3x(src.read()))
    print(f'Resampled {N_SAMPLES} tiles (seed={seed})')
    for f in sample_files:
        print(f'  {f}')

# Uncomment and run to get a fresh random set:
# resample(seed=123)